This section reads from the Delta raw table as a stream, parses the `body_str` JSON payload into explicit columns, keeps the raw `ingested_at` timestamp, and writes the result to `dev.bronze.ttc_trip_updates_raw_microbatch`.

In [0]:
from pyspark.sql import functions as F, types as T

source_table_microbatch = "dev.raw.ttc_trip_updates_raw_microbatch"
source_table_batch = "dev.raw.ttc_trip_updates_raw_batch"

target_table = "dev.bronze.ttc_trip_updates_bronze"
checkpoint_path = "/Volumes/dev/bronze/checkpoints/ttc_trip_updates_bronze"

ttc_trip_updates_schema = T.StructType([
    T.StructField("trip_id", T.StringType(), True),
    T.StructField("vehicle_id", T.StringType(), True),
    T.StructField("route_id", T.StringType(), True),
    T.StructField("schedule_relationship", T.StringType(), True),
    T.StructField("timestamp", T.LongType(), True),
    T.StructField("stop_sequence", T.IntegerType(), True),
    T.StructField("stop_id", T.StringType(), True),
    T.StructField("arrival_time", T.LongType(), True),      # Nullable timestamp
    T.StructField("departure_time", T.LongType(), True),    # Epoch timestamp
    T.StructField("id", T.StringType(), True)
])

In [0]:
raw_stream_microbatch = spark.readStream.table(source_table_microbatch)

bronze_stream_microbatch = (
    raw_stream_microbatch
    .select(
        F.from_json(F.col("data"), ttc_trip_updates_schema).alias("payload"),
        F.col("operation"),
        F.col("_ingest_ts").alias("ingested_at")
    )
    .select(
        F.col("operation"),
        F.col("payload.id").alias("event_id"),
        F.col("payload.trip_id").alias("trip_id"),
        F.col("payload.vehicle_id").alias("vehicle_id"),
        F.col("payload.route_id").alias("route_id"),
        F.col("payload.schedule_relationship").alias("schedule_relationship"),
        F.to_timestamp(F.from_unixtime(F.col("payload.timestamp"))).alias("event_timestamp"),
        F.col("payload.stop_sequence").alias("stop_sequence"),
        F.col("payload.stop_id").alias("stop_id"),
        F.to_timestamp(F.from_unixtime(F.col("payload.arrival_time"))).alias("arrival_time"),      # Nullable timestamp
        F.to_timestamp(F.from_unixtime(F.col("payload.departure_time"))).alias("departure_time"),    # Epoch timestamp
        F.col("ingested_at")
    )
)

In [0]:
raw_stream_batch = spark.readStream.table(source_table_batch)

bronze_stream_batch = (
    raw_stream_batch
    .select(
        F.from_json(F.col("data"), ttc_trip_updates_schema).alias("payload"),
        F.col("operation"),
        F.col("_ingest_ts").alias("ingested_at")
    )
    .select(
        F.col("operation"),
        F.col("payload.id").alias("event_id"),
        F.col("payload.trip_id").alias("trip_id"),
        F.col("payload.vehicle_id").alias("vehicle_id"),
        F.col("payload.route_id").alias("route_id"),
        F.col("payload.schedule_relationship").alias("schedule_relationship"),
        F.to_timestamp(F.from_unixtime(F.col("payload.timestamp"))).alias("event_timestamp"),
        F.col("payload.stop_sequence").alias("stop_sequence"),
        F.col("payload.stop_id").alias("stop_id"),
        F.to_timestamp(F.from_unixtime(F.col("payload.arrival_time"))).alias("arrival_time"),      # Nullable timestamp
        F.to_timestamp(F.from_unixtime(F.col("payload.departure_time"))).alias("departure_time"),    # Epoch timestamp
        F.col("ingested_at")
    )
)

In [0]:
bronze_stream_batch = bronze_stream_batch.withColumn(
    "hour_partition",
    F.date_trunc("hour", F.col("event_timestamp"))
)

bronze_stream_microbatch = bronze_stream_microbatch.withColumn(
    "hour_partition",
    F.date_trunc("hour", F.col("event_timestamp"))
)

def overwrite_existing_hours(df, batch_id):
    df_valid = df.filter(F.col("hour_partition").isNotNull())
    hours = [r["hour_partition"] for r in df_valid.select("hour_partition").distinct().collect()]
    if not hours:
        return

    predicates = " OR ".join(
        [f"hour_partition = TIMESTAMP '{h.strftime('%Y-%m-%d %H:%M:%S')}'" for h in hours]
    )

    (df.write
       .format("delta")
       .mode("overwrite")
       .option("replaceWhere", predicates)
       .saveAsTable(target_table))

In [0]:
batch_query = (
    bronze_stream_batch.writeStream
    .foreachBatch(overwrite_existing_hours)
    .option("checkpointLocation", checkpoint_path + "/batch")
    .start()
)

microbatch_query = (
    bronze_stream_microbatch.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path + "/microbatch")
    .toTable(target_table)
)